In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

print('Imported')

Imported


In [2]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [3]:
train_data = datasets.ImageFolder(root='./images/training', transform=transform)

test_data = datasets.ImageFolder(root='./images/testing', transform=transform)

In [4]:
type(train_data)

torchvision.datasets.folder.ImageFolder

In [5]:
print(train_data.classes)

print(test_data.classes)

['flip', 'notflip']
['flip', 'notflip']


In [6]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)

test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

In [7]:
image, label = train_data[0]

In [8]:
image.size()

torch.Size([3, 128, 128])

In [9]:
print(label)

0


In [10]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        #           channels,feature maps, kernel size 
        self.conv1 = nn.Conv2d(3, 12, 15) # 128 - 15 = 113 + 1 = 114x114 = (12, 114, 114)
        self.pool = nn.MaxPool2d(2,2) # (12, 57, 57)
        self.conv2 =  nn.Conv2d(12, 24, 16) # 57 - 16 = 41 + 1 = 42x42 = (24, 42, 42) -> Pooling (24, 21, 21)
        # Pooling
        self.conv3 = nn.Conv2d(24, 48, 16) # 21 - 16 = 5 + 1 = 6x6 = (48, 6, 6) -> Pooling (48, 3, 3)
        # Pooling
        self.fc1 = nn.Linear(48 * 3 * 3, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)

    def forward(self, X):
        X = self.pool(F.relu(self.conv1(X)))
        X = self.pool(F.relu(self.conv2(X)))
        X = self.pool(F.relu(self.conv3(X)))
        X = torch.flatten(X, 1)
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

In [11]:
net = NeuralNet()

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [12]:
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss =+ loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}', end='\n\n')

Training epoch 0...
Loss: 0.0092

Training epoch 1...
Loss: 0.0092

Training epoch 2...
Loss: 0.0083

Training epoch 3...
Loss: 0.0057

Training epoch 4...
Loss: 0.0058

Training epoch 5...
Loss: 0.0038

Training epoch 6...
Loss: 0.0069

Training epoch 7...
Loss: 0.0010

Training epoch 8...
Loss: 0.0020

Training epoch 9...
Loss: 0.0012

Training epoch 10...
Loss: 0.0013

Training epoch 11...
Loss: 0.0026

Training epoch 12...
Loss: 0.0009

Training epoch 13...
Loss: 0.0001

Training epoch 14...
Loss: 0.0004

Training epoch 15...
Loss: 0.0005

Training epoch 16...
Loss: 0.0001

Training epoch 17...
Loss: 0.0000

Training epoch 18...
Loss: 0.0007

Training epoch 19...
Loss: 0.0026

Training epoch 20...
Loss: 0.0008

Training epoch 21...
Loss: 0.0002

Training epoch 22...
Loss: 0.0004

Training epoch 23...
Loss: 0.0001

Training epoch 24...
Loss: 0.0012

Training epoch 25...
Loss: 0.0000

Training epoch 26...
Loss: 0.0000

Training epoch 27...
Loss: 0.0011

Training epoch 28...
Loss: 0.0

In [13]:
torch.save(net.state_dict(), 'trained_net.pth')

In [14]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [15]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100* correct/total

print(f'Accuracy: {accuracy}%')

Accuracy: 99.49748743718592%
